### Pytorch Dataset and DataLoader

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

DEVICE = (
    torch.device("mps")   if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Using device: {DEVICE}")

CSV_PATH = Path("../data/cleaned_metadata.csv")
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows  |  train={len(df[df.split=='train']):,}  test={len(df[df.split=='test']):,}")

Using device: mps
Loaded 31,936 rows  |  train=25,548  test=6,388


In [2]:
IMG_SIZE = 260  # EfficientNet-B2 native resolution

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [3]:
class ClothingDataset(Dataset):
    """
    Each item produces a 6-channel image (front + back concatenated along
    the channel dimension) and a dict of labels.

    Labels
    ------
    condition : long  0-4  (original 1-5 shifted to 0-indexed for CrossEntropyLoss)
    stains    : long  0-2
    holes     : long  0-2
    fraud     : float 0/1
    """

    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        front = Image.open(row["front_path"]).convert("RGB")
        back  = Image.open(row["back_path"]).convert("RGB")

        if self.transform:
            front = self.transform(front)
            back  = self.transform(back)

        img = torch.cat([front, back], dim=0)  # (6, H, W)

        labels = {
            "condition": torch.tensor(row["condition"] - 1, dtype=torch.long),
            "stains":    torch.tensor(row["stains"],        dtype=torch.long),
            "holes":     torch.tensor(row["holes"],         dtype=torch.long),
            "fraud":     torch.tensor(float(row["is_fraud_candidate"]), dtype=torch.float),
        }
        return img, labels

In [4]:
BATCH_SIZE = 32

train_df = df[df["split"] == "train"]
test_df  = df[df["split"] == "test"]

train_dataset = ClothingDataset(train_df, transform=train_transforms)
test_dataset  = ClothingDataset(test_df,  transform=test_transforms)

# Weighted sampler: oversample fraud candidates ~50x so the model
# sees them roughly as often as common items each epoch
fraud_weight = 50.0
sample_weights = train_df["is_fraud_candidate"].apply(
    lambda x: fraud_weight if x else 1.0
).values.copy()  # .copy() avoids non-writable array warning
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

# num_workers=0 because macOS + Jupyter can't pickle classes defined
# in notebook cells for multiprocessing. pin_memory not supported on MPS.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print(f"Train: {len(train_dataset):,} items → {len(train_loader):,} batches")
print(f"Test:  {len(test_dataset):,} items → {len(test_loader):,} batches")

Train: 25,548 items → 799 batches
Test:  6,388 items → 200 batches


In [5]:
# Sanity check: load one batch and inspect shapes + label ranges
imgs, labels = next(iter(train_loader))
print(f"Image batch shape : {imgs.shape}")        # (B, 6, 260, 260)
print(f"Condition range   : {labels['condition'].min().item()} – {labels['condition'].max().item()}")
print(f"Stains range      : {labels['stains'].min().item()} – {labels['stains'].max().item()}")
print(f"Holes range       : {labels['holes'].min().item()} – {labels['holes'].max().item()}")
print(f"Fraud in batch    : {labels['fraud'].sum().item():.0f} / {len(labels['fraud'])}")

Image batch shape : torch.Size([32, 6, 260, 260])
Condition range   : 0 – 4
Stains range      : 0 – 2
Holes range       : 0 – 2
Fraud in batch    : 4 / 32


### Model Architecture 

In [6]:
import timm

class ClothingMultiTask(nn.Module):
    """
    Multi-task model for clothing condition assessment.

    Backbone : EfficientNet-B2 (pretrained on ImageNet)
    Input    : 6-channel tensor (front RGB ∥ back RGB)
    Heads    : condition (5-class), stains (3-class), holes (3-class), fraud (binary)

    The first conv layer is widened from 3→6 input channels by duplicating
    the pretrained weights, so both views benefit from ImageNet features
    on day one.
    """

    def __init__(self, backbone_name: str = "efficientnet_b2", pretrained: bool = True):
        super().__init__()

        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            in_chans=3,
            num_classes=0,       # remove the default classifier head
            global_pool="avg",
        )

        # ── Widen first conv: 3 → 6 input channels ───────────────────────
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(
            6, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias is not None,
        )
        with torch.no_grad():
            # duplicate the pretrained 3-channel weights for both halves
            new_conv.weight[:, :3] = old_conv.weight
            new_conv.weight[:, 3:] = old_conv.weight
            if old_conv.bias is not None:
                new_conv.bias = old_conv.bias
        self.backbone.conv_stem = new_conv

        feat_dim = self.backbone.num_features  # 1408 for EfficientNet-B2

        # ── Task-specific heads ───────────────────────────────────────────
        self.head_condition = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 5),
        )
        self.head_stains = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 3),
        )
        self.head_holes = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 3),
        )
        self.head_fraud = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 1),
        )

    def forward(self, x):
        features = self.backbone(x)  # (B, feat_dim)
        return {
            "condition": self.head_condition(features),  # (B, 5)
            "stains":    self.head_stains(features),     # (B, 3)
            "holes":     self.head_holes(features),      # (B, 3)
            "fraud":     self.head_fraud(features).squeeze(-1),  # (B,)
        }

/opt/anaconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
model = ClothingMultiTask(pretrained=True).to(DEVICE)

# Quick architecture summary
total_params   = sum(p.numel() for p in model.parameters())
trainable      = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable:,}")
print(f"Backbone features: {model.backbone.num_features}")

# Verify forward pass with one batch
with torch.no_grad():
    test_out = model(imgs.to(DEVICE))
    for name, t in test_out.items():
        print(f"  {name:12s} → {t.shape}")

Total params     : 7,718,766
Trainable params : 7,718,766
Backbone features: 1408
  condition    → torch.Size([32, 5])
  stains       → torch.Size([32, 3])
  holes        → torch.Size([32, 3])
  fraud        → torch.Size([32])


### Training Loop

In [8]:
from tqdm import tqdm

# ── Loss functions ────────────────────────────────────────────────────────────
criterion_condition = nn.CrossEntropyLoss()
criterion_stains    = nn.CrossEntropyLoss()
criterion_holes     = nn.CrossEntropyLoss()
criterion_fraud     = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([50.0], device=DEVICE))

# task weights for the combined loss
LOSS_WEIGHTS = {"condition": 1.0, "stains": 0.5, "holes": 0.5, "fraud": 1.0}

def compute_losses(preds, labels):
    """Return per-task losses and the weighted total."""
    losses = {
        "condition": criterion_condition(preds["condition"], labels["condition"]),
        "stains":    criterion_stains(preds["stains"],       labels["stains"]),
        "holes":     criterion_holes(preds["holes"],         labels["holes"]),
        "fraud":     criterion_fraud(preds["fraud"],         labels["fraud"]),
    }
    total = sum(LOSS_WEIGHTS[k] * v for k, v in losses.items())
    return losses, total

In [9]:
LR          = 1e-4
NUM_EPOCHS  = 10
SAVE_DIR    = Path("../checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [10]:
# Sub-epoch logging: records loss & accuracy every ~5% of an epoch
step_log = {"step": [], "loss": [], "acc": []}

def train_one_epoch(model, loader, optimizer, epoch):
    model.train()
    running_loss = 0.0
    correct_cond = 0
    total = 0

    # window accumulators for sub-epoch snapshots
    window_loss = 0.0
    window_correct = 0
    window_total = 0

    num_batches = len(loader)
    log_every = max(1, num_batches // 20)  # ~20 checkpoints per epoch
    global_offset = (epoch - 1) * num_batches

    pbar = tqdm(loader, desc="  train", leave=False)
    for batch_idx, (imgs, labels) in enumerate(pbar):
        imgs = imgs.to(DEVICE)
        labels = {k: v.to(DEVICE) for k, v in labels.items()}

        preds = model(imgs)
        losses, total_loss = compute_losses(preds, labels)

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        bs = imgs.size(0)
        running_loss += total_loss.item() * bs
        correct_cond += (preds["condition"].argmax(1) == labels["condition"]).sum().item()
        total += bs

        window_loss += total_loss.item() * bs
        window_correct += (preds["condition"].argmax(1) == labels["condition"]).sum().item()
        window_total += bs

        if (batch_idx + 1) % log_every == 0 or (batch_idx + 1) == num_batches:
            step_log["step"].append(global_offset + batch_idx + 1)
            step_log["loss"].append(window_loss / window_total)
            step_log["acc"].append(window_correct / window_total)
            window_loss = 0.0
            window_correct = 0
            window_total = 0

        pbar.set_postfix(loss=f"{total_loss.item():.3f}")

    return running_loss / total, correct_cond / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    running_loss = 0.0
    correct_cond = 0
    total = 0
    fraud_tp = 0
    fraud_fp = 0
    fraud_fn = 0

    for imgs, labels in tqdm(loader, desc="  eval ", leave=False):
        imgs = imgs.to(DEVICE)
        labels = {k: v.to(DEVICE) for k, v in labels.items()}

        preds = model(imgs)
        _, total_loss = compute_losses(preds, labels)

        running_loss += total_loss.item() * imgs.size(0)
        correct_cond += (preds["condition"].argmax(1) == labels["condition"]).sum().item()

        fraud_pred = (preds["fraud"].sigmoid() >= 0.5).float()
        fraud_true = labels["fraud"]
        fraud_tp += ((fraud_pred == 1) & (fraud_true == 1)).sum().item()
        fraud_fp += ((fraud_pred == 1) & (fraud_true == 0)).sum().item()
        fraud_fn += ((fraud_pred == 0) & (fraud_true == 1)).sum().item()

        total += imgs.size(0)

    avg_loss = running_loss / total
    cond_acc = correct_cond / total
    fraud_prec = fraud_tp / (fraud_tp + fraud_fp) if (fraud_tp + fraud_fp) > 0 else 0.0
    fraud_rec  = fraud_tp / (fraud_tp + fraud_fn) if (fraud_tp + fraud_fn) > 0 else 0.0
    fraud_f1   = (2 * fraud_prec * fraud_rec / (fraud_prec + fraud_rec)
                  if (fraud_prec + fraud_rec) > 0 else 0.0)

    return avg_loss, cond_acc, fraud_prec, fraud_rec, fraud_f1

In [ ]:
history = {"train_loss": [], "train_acc": [],
           "val_loss": [], "val_acc": [],
           "fraud_prec": [], "fraud_rec": [], "fraud_f1": []}

best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}  (lr={optimizer.param_groups[0]['lr']:.2e})")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, epoch)
    val_loss, val_acc, f_prec, f_rec, f_f1 = evaluate(model, test_loader)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["fraud_prec"].append(f_prec)
    history["fraud_rec"].append(f_rec)
    history["fraud_f1"].append(f_f1)

    print(f"  train  loss={train_loss:.4f}  cond_acc={train_acc:.4f}")
    print(f"  val    loss={val_loss:.4f}  cond_acc={val_acc:.4f}")
    print(f"  fraud  prec={f_prec:.3f}  rec={f_rec:.3f}  f1={f_f1:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pt")
        print(f"  ✓ saved best model (val_loss={val_loss:.4f})")

torch.save(model.state_dict(), SAVE_DIR / "last_model.pt")
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")


Epoch 1/10  (lr=1.00e-04)


  train  loss=2.8433  cond_acc=0.4005
  val    loss=2.4231  cond_acc=0.3801
  fraud  prec=0.013  rec=0.174  f1=0.025
  ✓ saved best model (val_loss=2.4231)

Epoch 2/10  (lr=9.76e-05)


  train  loss=1.8959  cond_acc=0.4561
  val    loss=2.5940  cond_acc=0.3959
  fraud  prec=0.011  rec=0.174  f1=0.020

Epoch 3/10  (lr=9.05e-05)


  train  loss=1.7591  cond_acc=0.4707
  val    loss=2.5371  cond_acc=0.4044
  fraud  prec=0.062  rec=0.130  f1=0.085

Epoch 4/10  (lr=7.94e-05)


  train  loss=1.6385  cond_acc=0.4914
  val    loss=2.6114  cond_acc=0.4033
  fraud  prec=0.024  rec=0.130  f1=0.040

Epoch 5/10  (lr=6.55e-05)


  train  loss=1.5726  cond_acc=0.4992
  val    loss=2.6495  cond_acc=0.4250
  fraud  prec=0.034  rec=0.043  f1=0.038

Epoch 6/10  (lr=5.00e-05)


  train  loss=1.5627  cond_acc=0.5104
  val    loss=2.7332  cond_acc=0.4162
  fraud  prec=0.040  rec=0.043  f1=0.042

Epoch 7/10  (lr=3.45e-05)


  train  loss=1.5113  cond_acc=0.5116
  val    loss=2.6874  cond_acc=0.4123
  fraud  prec=0.033  rec=0.087  f1=0.048

Epoch 8/10  (lr=2.06e-05)


  train:  55%|█████▌    | 442/799 [06:54<05:39,  1.05it/s, loss=1.353]

Time to run: 

In [12]:
import json as _json

history_path  = SAVE_DIR / "training_history.json"
step_log_path = SAVE_DIR / "step_log.json"

with open(history_path, "w") as f:
    _json.dump(history, f, indent=2)
with open(step_log_path, "w") as f:
    _json.dump(step_log, f, indent=2)

print(f"Saved history  → {history_path}")
print(f"Saved step_log → {step_log_path}")

Saved history  → ../checkpoints/training_history.json
Saved step_log → ../checkpoints/step_log.json


In [13]:
import json as _json
from pathlib import Path as _Path

# Reload from disk if kernel was restarted and variables are empty
_save = _Path("../checkpoints")
if not history.get("train_loss"):
    with open(_save / "training_history.json") as f:
        history = _json.load(f)
    with open(_save / "step_log.json") as f:
        step_log = _json.load(f)
    print("Reloaded history & step_log from disk")
else:
    print("Using in-memory history (no reload needed)")

Using in-memory history (no reload needed)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Sub-epoch training curves (granular, every ~5% of an epoch) ───────────────
axes[0, 0].plot(step_log["step"], step_log["loss"], linewidth=0.8, color="steelblue")
axes[0, 0].set_title("Training Loss  (sub-epoch)")
axes[0, 0].set_xlabel("Batch step")
axes[0, 0].set_ylabel("Loss")
for e in range(1, NUM_EPOCHS + 1):
    axes[0, 0].axvline(x=e * len(train_loader), color="grey", linestyle="--", alpha=0.3)

axes[0, 1].plot(step_log["step"], step_log["acc"], linewidth=0.8, color="coral")
axes[0, 1].set_title("Training Condition Accuracy  (sub-epoch)")
axes[0, 1].set_xlabel("Batch step")
axes[0, 1].set_ylabel("Accuracy")
for e in range(1, NUM_EPOCHS + 1):
    axes[0, 1].axvline(x=e * len(train_loader), color="grey", linestyle="--", alpha=0.3)

# ── Per-epoch train vs val ────────────────────────────────────────────────────
epochs = range(1, len(history["train_loss"]) + 1)

axes[1, 0].plot(epochs, history["train_loss"], "o-", label="train", color="steelblue")
axes[1, 0].plot(epochs, history["val_loss"],   "o-", label="val",   color="coral")
axes[1, 0].set_title("Loss per Epoch")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Loss")
axes[1, 0].legend()

axes[1, 1].plot(epochs, history["train_acc"], "o-", label="train", color="steelblue")
axes[1, 1].plot(epochs, history["val_acc"],   "o-", label="val",   color="coral")
axes[1, 1].set_title("Condition Accuracy per Epoch")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Accuracy")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# ── Fraud detection metrics over epochs ───────────────────────────────────────
fig2, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, history["fraud_prec"], "o-", label="Precision", color="mediumseagreen")
ax.plot(epochs, history["fraud_rec"],  "o-", label="Recall",    color="coral")
ax.plot(epochs, history["fraud_f1"],   "s-", label="F1",        color="steelblue", linewidth=2)
ax.set_title("Fraud Detection Metrics per Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

### Evaluation and Reporting